# 📖 AudiobookMaker Google Colab WebUI

Welcome to the Google Colab Notebook for **AudiobookMaker**! This notebook sets up and runs the AudiobookMaker application.

Since AudiobookMaker uses both Python and a high-performance Rust PyO3 acceleration module (`audiobook_rust`), this notebook will:
1. Validate environment configuration (GPU runtime).
2. Clone/locate the project repository.
3. Install the Rust compiler toolchain.
4. Compile the Rust PyO3 modules on Google Colab.
5. Install all Python dependencies (including PyTorch with CUDA & `qwen-tts`).
6. Launch the FastAPI orchestration server in the background.
7. Launch the Gradio Web UI with a public shareable link.

### ⚡ Select GPU Runtime (CRITICAL)
Before running any cells, make sure your Google Colab session has a GPU assigned:
- Open the top menu: **Runtime** -> **Change runtime type**.
- Set **Hardware accelerator** to **T4 GPU** (or A100/V100 if available).
- Click **Save**.

In [ ]:
# Check if GPU accelerator is assigned
import subprocess
try:
    gpu_info = subprocess.check_output("nvidia-smi").decode("utf-8")
    print(gpu_info)
    print("✅ GPU Accelerator is correctly assigned!")
except FileNotFoundError:
    print("\n" + "="*80)
    print("⚠️ WARNING: GPU accelerator not detected (nvidia-smi not found)!")
    print("The application will run, but text-to-speech generation will be EXTREMELY slow.")
    print("To enable GPU:")
    print("  1. Go to top menu: Runtime -> Change runtime type.")
    print("  2. Select 'T4 GPU' under Hardware accelerator.")
    print("="*80 + "\n")

### 📂 Clone Repository or Setup Workspace

In [ ]:
import os

# Google Colab working directory is /content
COLAB_WORKING = '/content'
os.chdir(COLAB_WORKING)

# Check if we are already in a repository root
if os.path.exists("audiobook_rust") and os.path.exists("requirements.txt"):
    print("Already in the AudiobookMaker repository root directory:", os.getcwd())
elif os.path.exists("AudiobookMaker/audiobook_rust"):
    print("Found AudiobookMaker directory. Changing directory...")
    os.chdir("AudiobookMaker")
    print("Current directory:", os.getcwd())
else:
    print("Cloning AudiobookMaker repository...")
    clone_status = os.system("git clone https://github.com/MSpider3/AudiobookMaker.git")
    if clone_status != 0:
        raise RuntimeError("Failed to clone the repository!")
    
    if os.path.exists("AudiobookMaker"):
        os.chdir("AudiobookMaker")
        print("Successfully cloned and entered AudiobookMaker directory.")
    else:
        raise FileNotFoundError("Could not find the cloned AudiobookMaker directory!")

# Final check
if not os.path.exists("audiobook_rust"):
    raise FileNotFoundError("Invalid workspace: 'audiobook_rust' directory not found at " + os.getcwd())

### 🦀 Install Rust Compiler Toolchain

In [ ]:
print("Installing Rustup and Cargo...")
install_rust = os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
if install_rust != 0:
    print("Warning: Rust installation script exited with non-zero status.")

# Add Cargo binaries to Python PATH env variable
import os
cargo_bin = os.path.expanduser("~/.cargo/bin")
if cargo_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + cargo_bin

# Verify Rust tools are available
print("\nVerifying Rust compiler installation:")
cargo_ver = os.system("cargo --version")
rustc_ver = os.system("rustc --version")

if cargo_ver != 0 or rustc_ver != 0:
    print("⚠️ WARNING: Rust compiler or Cargo not found in PATH.")
    print("Make sure you add ~/.cargo/bin to your PATH.")

### 📦 Install System Dependencies & Python Requirements

In [ ]:
print("Updating system packages and installing FFmpeg...")
os.system("apt-get update && apt-get install -y ffmpeg")

print("\nInstalling Python dependencies (this will take a few minutes)...")
pip_status = os.system("pip install -r requirements.txt")
if pip_status != 0:
    print("Warning: pip install -r requirements.txt returned a non-zero status.")

print("\nInstalling maturin (needed to build python-rust PyO3 modules)...")
os.system("pip install maturin")

### ⚙️ Compile Rust PyO3 Module

In [ ]:
import os

print("Compiling audiobook_rust module using maturin...")
current_dir = os.getcwd()
try:
    os.chdir('audiobook_rust')
    build_status = os.system("pip install .")
    if build_status != 0:
        print("Warning: maturin build returned a non-zero status.")
finally:
    os.chdir(current_dir)

# Verify import works
try:
    import importlib
    importlib.invalidate_caches()
    import audiobook_rust
    print("🎉 Success! audiobook_rust compiled and imported successfully. 5.5x faster mastering is enabled.")
except ImportError as e:
    print("\n" + "="*80)
    print("⚠️ WARNING: Error compiling Rust code:", e)
    print("Fallback: The app will run, but with slower pure-Python fallback code.")
    print("="*80 + "\n")

### 🚀 Start FastAPI Orchestration Server in Background

In [ ]:
import subprocess
import time
import requests

print("Starting start_api.py in the background...")
api_proc = subprocess.Popen(["python", "start_api.py"])

print("Waiting for FastAPI server to start...")
for i in range(45):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1.0)
        if r.status_code == 200 and r.json().get("status") == "ok":
            print(f"\n✅ FastAPI Server is healthy and running after {i+1} seconds!")
            break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n⚠️ Warning: FastAPI healthcheck timed out. Python Gradio UI will use local processing fallback.")

### 🎨 Launch Gradio Web Interface & Tunnel
Run this cell to start the UI. This cell will:
1. Start the Gradio Web UI locally on the Colab environment.
2. Establish a **public tunnel** using Pinggy with automatic renewal.

In [ ]:
from app import build_app
import subprocess
import time
import re
import threading

demo = build_app()

try:
    print("1. Launching Gradio interface on local port 7860...")
    demo.launch(
        server_name="127.0.0.1",
        server_port=7860,
        share=False,
        prevent_thread_lock=True
    )
    print("✅ Gradio local server started successfully!")
except Exception as e:
    print(f"❌ Error starting Gradio: {e}")

RENEW_INTERVAL_SECONDS = 50 * 60
SSH_CMD = [
    "ssh", "-p", "443",
    "-R", "0:localhost:7860",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "-o", "IdentitiesOnly=yes",
    "-F", "/dev/null",
    "free.pinggy.io"
]
URL_RE = re.compile(r"https?://[a-zA-Z0-9.-]+\.pinggy[a-zA-Z0-9.-]+")
_tunnel_stop = threading.Event()

def _start_tunnel():
    proc = subprocess.Popen(SSH_CMD, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    urls = []
    def _reader(p, u):
        for line in iter(p.stdout.readline, ''):
            if "pinggy" in line and "dashboard" not in line:
                m = URL_RE.search(line)
                if m:
                    u.append(m.group(0))
    t = threading.Thread(target=_reader, args=(proc, urls), daemon=True)
    t.start()
    return proc, urls, t

def _tunnel_manager():
    generation = 0
    while not _tunnel_stop.is_set():
        generation += 1
        print(f"\n[Tunnel] Starting tunnel (generation {generation})...")
        try:
            proc, urls, _ = _start_tunnel()
        except FileNotFoundError:
            print("[Tunnel] ❌ ssh not found. Cannot create tunnel.")
            return
        except Exception as e:
            print(f"[Tunnel] ❌ Error: {e}")
            return

        for _ in range(10):
            if urls or _tunnel_stop.is_set():
                break
            time.sleep(1)

        if urls:
            print("\n" + "="*80)
            if generation == 1:
                print("🎉 PUBLIC ACCESS LINKS READY!")
            else:
                print(f"🔄 TUNNEL RENEWED (generation {generation}) — old link has expired. Use the new link:")
            for url in set(urls):
                print(f"👉 {url}")
            print("="*80)
        else:
            print("[Tunnel] ⚠️ No URL received. Check connectivity.")

        _tunnel_stop.wait(timeout=RENEW_INTERVAL_SECONDS)
        try:
            proc.terminate()
            proc.wait(timeout=5)
        except Exception:
            pass

print("\n2. Starting self-renewing public tunnel via Pinggy...")
manager_thread = threading.Thread(target=_tunnel_manager, daemon=True)
manager_thread.start()

print("\nGeneration is running. The tunnel will auto-renew every 50 minutes.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping web server and tunnel...")
    _tunnel_stop.set()
    manager_thread.join(timeout=10)
    demo.close()

### ⚡ Headless CLI Generation (Alternative to Gradio)

**Prefer the CLI for faster, more reliable generation — especially on long sessions where the Gradio tunnel may expire.**

#### Workflow:
1. Use the **Gradio cell above** to configure all settings (voice, chapters, TTS model, etc.).
2. In the Gradio UI, click **'📋 Export Config JSON'** to save your settings and all chapter text into a single `generation_progress.json` file.
3. Download the JSON file from Gradio (or find it in `audiobook_output/<BookTitle>/generation_progress.json`).
4. Fill in the paths/links below and run this cell.

**Supported input sources:**
- Upload files to `/content/` via the Colab files sidebar.
- Paste a Google Drive shareable link (auto-downloaded via `gdown`).
- Leave `BOOK_FILE` / `VOICE_FILE` / `COVER_FILE` blank if JSON already has cached data.

In [ ]:
import os, subprocess

# ── Fill in your file paths or Google Drive / HTTP links ─────────────────────
CONFIG_JSON = ""  # Required: path or link to generation_progress.json
BOOK_FILE   = ""  # Optional: path or link to book file (.epub, .pdf, .mobi, etc.)
VOICE_FILE  = ""  # Required: path or link to narrator voice (.wav)
COVER_FILE  = ""  # Optional: path or link to cover image (.jpeg, .jpg, .png, .webp)

# ── Auto-download helper ──────────────────────────────────────────────────────
def resolve_file(path_or_url, default_name="file"):
    if not path_or_url:
        return ""
    if "drive.google.com" in path_or_url:
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
        dl = gdown.download(path_or_url, quiet=False, fuzzy=True)
        if dl and os.path.exists(dl):
            return os.path.abspath(dl)
        out = f"/content/{default_name}"
        return out if os.path.exists(out) else path_or_url
    elif path_or_url.startswith("http"):
        import urllib.parse
        parsed = urllib.parse.urlparse(path_or_url)
        filename = os.path.basename(parsed.path) or default_name
        out = f"/content/{filename}"
        subprocess.run(["wget", "-q", "-O", out, path_or_url], check=True)
        return out
    return path_or_url

config_path = resolve_file(CONFIG_JSON, "generation_progress.json")
book_path   = resolve_file(BOOK_FILE,   "book_file")
voice_path  = resolve_file(VOICE_FILE,  "narrator_voice.wav")
cover_path  = resolve_file(COVER_FILE,  "cover_image.jpg")

if not config_path or not os.path.exists(config_path):
    raise FileNotFoundError("CONFIG_JSON is required. Fill it in above.")

# ── Build and run CLI command ─────────────────────────────────────────────────
cmd = f'python cli.py "{config_path}"'
if book_path:
    cmd += f' --book-path "{book_path}"'
if voice_path:
    cmd += f' --voice-file "{voice_path}"'
if cover_path:
    cmd += f' --cover-image "{cover_path}"'

print(f"Running: {cmd}\n")
os.system(cmd)